# Colab Training Notebook for Iris Segmentation

Notebook này clone code từ GitHub, train trên Google Colab, và lưu checkpoint/model vào Google Drive để an toàn khi session bị ngắt.

## 1. Mount Google Drive để lưu checkpoint

Mount Drive trước để mọi checkpoint, log, và model cuối cùng được lưu an toàn.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/iris-segformer')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
LOG_DIR = DRIVE_ROOT / 'logs'
MODEL_DIR = DRIVE_ROOT / 'models'
VIS_DIR = DRIVE_ROOT / 'visualizations'
for folder in [DRIVE_ROOT, CHECKPOINT_DIR, LOG_DIR, MODEL_DIR, VIS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Google Drive mounted at:', DRIVE_ROOT)


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/<your-username>/<your-repo>.git'
REPO_DIR = Path('/content/Eris')

if REPO_DIR.exists():
    print('Repo already exists:', REPO_DIR)
else:
    !git clone $REPO_URL /content/Eris
    print('Cloned repo to:', REPO_DIR)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())


## 2. Kết nối và clone repo từ GitHub

Clone repository từ GitHub vào Colab để lấy source code, scripts, và notebook.

In [ ]:
!pip install -r requirements.txt

print('Installed dependencies from requirements.txt')


## 3. Cài đặt thư viện cần thiết trên Colab

Cài dependencies từ requirements.txt của repo trước khi train.

In [ ]:
import os
import subprocess
import sys

# If you use the Kaggle mirror, set these before running the download script.
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'

subprocess.run([sys.executable, 'scripts/download_ubiris.py'], cwd=str(REPO_DIR), check=True)
subprocess.run(
    [sys.executable, 'scripts/prepare_dataset.py', '--source-images', 'dataset/raw/images', '--source-masks', 'dataset/raw/masks', '--copy'],
    cwd=str(REPO_DIR),
    check=True,
)
print('Dataset downloaded and prepared.')


## 4. Tải dữ liệu và cấu hình từ GitHub

Chạy các script trong repo để tải UBIRIS V2 và chuẩn bị lại thư mục dataset cho Colab.

In [ ]:
from pathlib import Path
import yaml

BASE_CONFIG = REPO_DIR / 'configs' / 'train.yaml'
COLAB_CONFIG = DRIVE_ROOT / 'train_colab.yaml'

config = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
config['project']['output_dir'] = str(DRIVE_ROOT)
config['project']['name'] = 'iris-segmentation-colab'
COLAB_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

print('Saved Colab config to:', COLAB_CONFIG)
print('All outputs will be stored under:', DRIVE_ROOT)


## 5. Thiết lập đường dẫn lưu model trên Drive

Đặt đường dẫn trỏ tới Google Drive để lưu checkpoint, best model, và file trạng thái train.

In [ ]:
import torch

from src.losses import CombinedDiceFocalLoss
from src.models import SegFormerCustom

model_cfg = config['model']
train_cfg = config['training']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SegFormerCustom(
    backbone_name=model_cfg['backbone_name'],
    num_classes=model_cfg['num_classes'],
    decoder_channels=model_cfg.get('decoder_channels', 256),
    dropout=model_cfg.get('dropout', 0.1),
)
model.to(device)

criterion = CombinedDiceFocalLoss(
    dice_weight=config['loss'].get('dice_weight', 0.6),
    focal_weight=config['loss'].get('focal_weight', 0.4),
    focal_alpha=config['loss'].get('focal_alpha', 0.8),
    focal_gamma=config['loss'].get('focal_gamma', 2.0),
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg.get('lr', 1e-4),
    weight_decay=train_cfg.get('weight_decay', 1e-5),
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_cfg.get('epochs', 2))

print('Device:', device)
print('Model initialized:', model.__class__.__name__)
print('Optimizer:', optimizer.__class__.__name__)
print('Scheduler:', scheduler.__class__.__name__)


## 6. Khởi tạo mô hình và tham số train

Notebook sẽ dùng config đã chỉnh để train từ repo GitHub, còn checkpoint vẫn lưu vào Google Drive.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'train.py', '--config', str(COLAB_CONFIG)], cwd=str(REPO_DIR), check=True)
print('Training finished. Checkpoints should now be in Google Drive.')


## 7. Train model và lưu checkpoint định kỳ vào Drive

Cell này chạy training bình thường và lưu `best.pt`, `latest.pt`, `epoch_summary.txt`, `metrics.csv` vào Google Drive.

In [ ]:
import torch

latest_ckpt = CHECKPOINT_DIR / 'latest.pt'
best_ckpt = CHECKPOINT_DIR / 'best.pt'

for path in [latest_ckpt, best_ckpt]:
    if path.exists():
        checkpoint = torch.load(path, map_location='cpu')
        print(f'Found checkpoint: {path.name}')
        print('Keys:', list(checkpoint.keys()))
        print('Epoch:', checkpoint.get('epoch'))
    else:
        print(f'Not found: {path.name}')

print('Keep the latest checkpoint so you can inspect or reuse it later.')


## 8. Khôi phục training từ checkpoint đã lưu

Nếu Colab bị ngắt, dùng cell này để kiểm tra checkpoint gần nhất trên Google Drive trước khi chạy tiếp.

In [ ]:
from pathlib import Path

files_to_check = [
    CHECKPOINT_DIR / 'best.pt',
    CHECKPOINT_DIR / 'latest.pt',
    DRIVE_ROOT / 'epoch_summary.txt',
    DRIVE_ROOT / 'metrics.csv',
    COLAB_CONFIG,
]

for path in files_to_check:
    if path.exists():
        print(f'[OK] {path} | {path.stat().st_size} bytes')
    else:
        print(f'[MISSING] {path}')

!ls -lah /content/drive/MyDrive/iris-segformer


## 9. Kiểm tra file model đã lưu an toàn

Sau khi train xong, chạy cell này để xác minh checkpoint và model final đã nằm trong Google Drive.

In [ ]:
from pathlib import Path

files_to_check = [
    CHECKPOINT_DIR / 'best.pt',
    CHECKPOINT_DIR / 'latest.pt',
    DRIVE_ROOT / 'epoch_summary.txt',
    DRIVE_ROOT / 'metrics.csv',
    COLAB_CONFIG,
]

for path in files_to_check:
    if path.exists():
        print(f'[OK] {path} | {path.stat().st_size} bytes')
    else:
        print(f'[MISSING] {path}')

!ls -lah /content/drive/MyDrive/iris-segformer
